In [1]:
#import required library
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,Dataset
import torch.optim as optim

In [2]:
# Set random seeds for reproducibility
torch.manual_seed(42)

In [3]:
df = pd.read_csv('fmnist_small.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [4]:
x= df.iloc[:,1:].values

In [5]:
y=df.iloc[:,0].values

In [6]:
y.shape

(6000,)

In [7]:
x.shape

(6000, 784)

In [44]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=43)
print(x_train.shape,y_train.shape,x_test.shape,y_test.shape)

(4800, 784) (4800,) (1200, 784) (1200,)


In [45]:
print(x_train.shape,y_train.shape,x_test.shape,y_test.shape)

(4800, 784) (4800,) (1200, 784) (1200,)


In [46]:
x_train =x_train/255.0
x_test = x_test/255.0

In [47]:
x_train.shape

(4800, 784)

In [48]:
x_test.shape

(1200, 784)

In [52]:
y_test.shape

(1200,)

In [53]:
class CusomDataSet(Dataset):
    def __init__(self,features,labels):
        self.features = torch.tensor(features,dtype=torch.float32)
        self.labels = torch.tensor(labels,dtype=torch.long)

    def __len__(self):
        
        return len(self.features)
    def __getitem__(self, index):
        
        return self.features[index],self.labels[index]

In [72]:
class MyAnn(nn.Module):
    def __init__(self, n_features):
        super().__init__()

        self.Linear = nn.Sequential(
            nn.Linear(n_features,128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(128,64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(64,10)
        )
    def forward(self,x):

        return self.Linear(x)


In [76]:
train_dataset = CusomDataSet(x_train,y_train)
test_dataset = CusomDataSet(x_test,y_test)

In [75]:
len(train_dataset)

4800

In [77]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False)

In [78]:
epochs =100
lr = 0.1

In [79]:
Model = MyAnn(x_train.shape[1])

loss_function = nn.CrossEntropyLoss()

optimizer = optim.SGD(Model.parameters(),lr=lr)

In [80]:
total =0
for epoch in range(epochs):
    for batch_features,batch_labels in train_loader:

        outputs = Model(batch_features)

        loss = loss_function(outputs,batch_labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

    print(f"epoch : {epoch+1} , losss : {loss.item()}")

epoch : 1 , losss : 0.7841830253601074
epoch : 2 , losss : 0.6051285266876221
epoch : 3 , losss : 0.578913688659668
epoch : 4 , losss : 0.9275380373001099
epoch : 5 , losss : 0.459045946598053
epoch : 6 , losss : 0.6287808418273926
epoch : 7 , losss : 0.35544154047966003
epoch : 8 , losss : 0.22424781322479248
epoch : 9 , losss : 0.3403896391391754
epoch : 10 , losss : 0.4047531187534332
epoch : 11 , losss : 0.5368958711624146
epoch : 12 , losss : 0.3632217049598694
epoch : 13 , losss : 0.4443831443786621
epoch : 14 , losss : 0.49882930517196655
epoch : 15 , losss : 0.2212471216917038
epoch : 16 , losss : 0.28968295454978943
epoch : 17 , losss : 0.320407509803772
epoch : 18 , losss : 0.33578920364379883
epoch : 19 , losss : 0.34762322902679443
epoch : 20 , losss : 0.2567022740840912
epoch : 21 , losss : 0.1933136135339737
epoch : 22 , losss : 0.21863070130348206
epoch : 23 , losss : 0.2549263834953308
epoch : 24 , losss : 0.21776771545410156
epoch : 25 , losss : 0.1996796727180481
epoc

In [81]:
Model.eval()

MyAnn(
  (Linear): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [82]:
# evaluation code
total = 0
correct = 0

with torch.no_grad():

  for test_features, test_labels in test_loader:

    outputs = Model(test_features)
    _, predicted = torch.max(outputs, 1)

    total = total + test_labels.shape[0]

    correct = correct + (predicted == test_labels).sum().item()

print(correct/total)


0.8316666666666667


In [83]:
total = 0
curect = 0
for train_features,train_labels in train_loader:

    outputs=Model(train_features)

    loss = loss_function(outputs,train_labels)

    _,predicted = torch.max(outputs,1)

    total = total + train_features.shape[0]

    curect = curect + (predicted==train_labels).sum().item()
print(curect/total)



0.9947916666666666
